In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Order Items Profiling").getOrCreate()
order_items = spark.read.parquet("../data/bronze/order_items")

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)



In [3]:
order_items.count()

112650

In [4]:
order_items.show(10, truncate=False)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price |freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+
|8ac26cb701a7887cc7c458683408c56a|1            |4ebb87ba41ca446326804d0e7f8dcc93|7a67c85e85bb2ce8582c35f2203ad736|2017-05-22 16:05:14|109.99|18.02        |
|8ac2728285fd4228f35d907768b5288a|1            |8b90be4893a4277a9f33c5b2348cf9c6|004c9cd9d87a3c30c522c48c4fc07416|2017-03-15 14:09:17|109.99|8.27         |
|8ac2728285fd4228f35d907768b5288a|2            |fa94f25a73969e3a25d52706e246dfcc|004c9cd9d87a3c30c522c48c4fc07416|2017-03-15 14:09:17|109.99|16.55        |
|8ac2728285fd4228f35d907768b5288a|3            |b01cedfa96d89142

In [5]:
from pyspark.sql.functions import col, when, count

In [6]:
order_items.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in order_items.columns
]).show(vertical=True)

-RECORD 0------------------
 order_id            | 0   
 order_item_id       | 0   
 product_id          | 0   
 seller_id           | 0   
 shipping_limit_date | 0   
 price               | 0   
 freight_value       | 0   



In [10]:
duplicate_order_items = order_items.groupBy("order_id","order_item_id").count().filter(col("count")>1)
print("Duplicate order items:", duplicate_order_items.count())

Duplicate order items: 0


In [11]:
order_items.select("price", "freight_value").show(10)

+------+-------------+
| price|freight_value|
+------+-------------+
|109.99|        18.02|
|109.99|         8.27|
|109.99|        16.55|
|259.99|        21.01|
|109.99|        16.55|
| 65.99|        11.48|
|599.65|        18.95|
| 28.90|        34.15|
| 74.90|        15.27|
| 74.90|        15.27|
+------+-------------+
only showing top 10 rows


In [12]:
order_items.select("shipping_limit_date").show(10, truncate=False)

+-------------------+
|shipping_limit_date|
+-------------------+
|2017-05-22 16:05:14|
|2017-03-15 14:09:17|
|2017-03-15 14:09:17|
|2017-03-15 14:09:17|
|2017-03-15 14:09:17|
|2018-06-28 04:58:48|
|2017-12-14 10:10:48|
|2018-01-23 08:59:38|
|2018-02-14 01:35:40|
|2018-02-14 01:35:40|
+-------------------+
only showing top 10 rows
